In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(497128, 31)


In [2]:
df["hubName"].value_counts()

hubName
Instant Foods(Noida)             109133
Instant Foods (SED)               64403
Instant Foods (GZB)               49620
Rapid Enterprises                 49610
Crossline Events (Noida)          46815
Crossline Events (Meerut)         46558
Cross Line Events (Ghaziabad)     42527
Cross Line Events (EDelhi)        40769
NB Enterprises (West Delhi)       28644
Instant Foods (Mathura)           17836
Maa Sharda(LKO)                    1213
Name: count, dtype: int64

In [3]:
region_trends = (
    df.groupby(
        [
            "hubId",
            "hubName",
            "shopType",
            "skuNumber"
        ]
    )
    .agg(
        total_qty=(
            "effective_qty",
            "sum"
        ),

        unique_buyers=(
            "customerId",
            "nunique"
        ),

        total_orders=(
            "orderNumber",
            "nunique"
        )
    )
    .reset_index()
)

region_trends.head()

,hubId,hubName,shopType,skuNumber,total_qty,unique_buyers,total_orders
0,HB002,Instant Foods(Noida),Chemist A,SKU00010,1,1,1
1,HB002,Instant Foods(Noida),Chemist A,SKU00062,1,1,1
2,HB002,Instant Foods(Noida),Chemist A,SKU479,1,1,1
3,HB002,Instant Foods(Noida),Chemist A,SKU501,1,1,1
4,HB002,Instant Foods(Noida),Chemist A,SKU544,1,1,1


In [4]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category"
        ]
    ]
    .drop_duplicates()
)

region_trends = (
    region_trends.merge(
        sku_lookup,
        on="skuNumber",
        how="left"
    )
)

region_trends.head()

,hubId,hubName,shopType,skuNumber,total_qty,unique_buyers,total_orders,itemName,brand,category
0,HB002,Instant Foods(Noida),Chemist A,SKU00010,1,1,1,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners
1,HB002,Instant Foods(Noida),Chemist A,SKU00062,1,1,1,RG Pearl Elaichi MP Pouch 0.14g,DS Group,Mouth Fresheners
2,HB002,Instant Foods(Noida),Chemist A,SKU479,1,1,1,Haldiram - Punjabi Tadka (24 pcs),Haldiram,Namkeen
3,HB002,Instant Foods(Noida),Chemist A,SKU501,1,1,1,Haldiram - Aloo Bhujia (24 pcs),Haldiram,Namkeen
4,HB002,Instant Foods(Noida),Chemist A,SKU544,1,1,1,Pinata - Fruit Lollipops,LuvIt,LuvIt


In [5]:
region_trends[
    "region_rank"
] = (
    region_trends
    .groupby(
        [
            "hubId",
            "shopType"
        ]
    )
    ["unique_buyers"]
    .rank(
        ascending=False,
        method="dense"
    )
)

region_trends.head()

,hubId,hubName,shopType,skuNumber,total_qty,unique_buyers,total_orders,itemName,brand,category,region_rank
0,HB002,Instant Foods(Noida),Chemist A,SKU00010,1,1,1,Pass Pass Meetha Magic 11.75g Hanger,Pass Pass,Mouth Fresheners,1.0
1,HB002,Instant Foods(Noida),Chemist A,SKU00062,1,1,1,RG Pearl Elaichi MP Pouch 0.14g,DS Group,Mouth Fresheners,1.0
2,HB002,Instant Foods(Noida),Chemist A,SKU479,1,1,1,Haldiram - Punjabi Tadka (24 pcs),Haldiram,Namkeen,1.0
3,HB002,Instant Foods(Noida),Chemist A,SKU501,1,1,1,Haldiram - Aloo Bhujia (24 pcs),Haldiram,Namkeen,1.0
4,HB002,Instant Foods(Noida),Chemist A,SKU544,1,1,1,Pinata - Fruit Lollipops,LuvIt,LuvIt,1.0


In [6]:
(
    region_trends[
        region_trends["hubName"]
        == "Instant Foods(Noida)"
    ]
    .sort_values(
        "region_rank"
    )
    [
        [
            "itemName",
            "unique_buyers",
            "region_rank"
        ]
    ]
    .head(10)
)

,itemName,unique_buyers,region_rank
0,Pass Pass Meetha Magic 11.75g Hanger,1,1.0
1,RG Pearl Elaichi MP Pouch 0.14g,1,1.0
34,Crax Crunchy Pipes,1,1.0
35,Crax Natkhat,1,1.0
39,Fun Flips Cheesy Pizza,1,1.0
38,Fun Flips Puffs - Pudina,1,1.0
37,Fun Flips Puffs - Khatta Meetha,1,1.0
36,Fun Flips Mocktail Curly Puffs,1,1.0
30,Cadbury 5 Star ( RS. 5),1,1.0
29,Cadbury Dairy Milk ( Rs. 10),1,1.0


In [7]:
region_trends.to_parquet(
    "../data/processed/region_trends.parquet",
    index=False
)

print("Saved")

Saved


In [8]:
print(region_trends.shape)

print(
    region_trends["hubName"]
    .nunique()
)

print(
    region_trends["shopType"]
    .nunique()
)

(10526, 11)
11
10


In [9]:
region_trends.shape

(10526, 11)

In [10]:
region_trends.sort_values(
    "region_rank"
)[
    [
        "hubName",
        "shopType",
        "itemName",
        "region_rank"
    ]
].head(15)

,hubName,shopType,itemName,region_rank
1,Instant Foods(Noida),Chemist A,RG Pearl Elaichi MP Pouch 0.14g,1.0
0,Instant Foods(Noida),Chemist A,Pass Pass Meetha Magic 11.75g Hanger,1.0
10041,Instant Foods (Mathura),Paan A,Rajnigandha 4g,1.0
10040,Instant Foods (Mathura),Paan A,Rajnigandha 4g,1.0
543,Instant Foods(Noida),General C,Rajnigandha 4g,1.0
542,Instant Foods(Noida),General C,Rajnigandha 4g,1.0
4147,Cross Line Events (Ghaziabad),Paan A,Tulsi 00 (with Silver) 0.45g,1.0
4146,Cross Line Events (Ghaziabad),Paan A,TZ 00 (with Silver) 0.45g,1.0
4283,Cross Line Events (Ghaziabad),Paan B,Tulsi 00 (with Silver) 0.45g,1.0
4282,Cross Line Events (Ghaziabad),Paan B,TZ 00 (with Silver) 0.45g,1.0


In [11]:
region_trends.shape

(10526, 11)